# Variant 3 — Deeper / Wider ATN and STN

Widens the analysis (ATN) and synthesis (STN) transforms from 3 layers × 16 channels to 4 layers × 32 channels. The new mid layer keeps stride (1,1) so it just refines features before the final compression.

This notebook is a standalone, simplified version of `var3_deeper_atn_stn.py` and follows the same flow as `4 feb/ADJSCC-CSInet+.ipynb`:
1. dataset
2. AF module
3. ATN module
4. encoder
5. real → complex symbols + power normalisation
6. wireless channel
7. complex → real (C2R)
8. decoder
9. STN
10. training loop


## Imports and seed

In [ ]:
import math
import os
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Config

In [ ]:
class TrainingConfig:
    train_file:        str   = "train_data.mat"
    val_file:          str   = "val_data.mat"
    test_file:         str   = "test_data.mat"
    checkpoint_dir:    str   = "checkpoints_deeper_atn_stn"
    fast_dev_run:      bool  = False
    run_training:      bool  = False
    epochs:            int   = 500
    batch_size:        int   = 200
    learning_rate:     float = 0.001
    min_lr:            float = 0.0001
    patience:          int   = 20
    weight_decay:      float = 1e-5
    grad_clip:         float = 1.0
    snr_low:           float = -10.0
    snr_high:          float = 10.0
    k_feedback:        int   = 64
    compression_ratio: int   = 16
    mse_weight:        float = 0.5
    nmse_weight:       float = 0.5
    warmup_epochs:     int   = 20
    save_every:        int   = 10
    run_evaluation:    bool  = False
    checkpoint_path:   str | None = None
    resume_latest:     bool  = False
    # --- NEW: ATN/STN architecture parameters ---
    atn_channels:      int   = 32   # hidden channels (was 16)
    atn_layers:        int   = 4    # conv layers in ATN (was 3)
cfg = TrainingConfig()
os.makedirs(cfg.checkpoint_dir, exist_ok=True)


## Dataset

Streams the QuaDRiGa CSI HDF5 files lazily and applies a global per-channel scale computed on the train split.

In [ ]:
def _read_scalar(dataset):
    value = dataset[()]
    if isinstance(value, np.ndarray) and value.size == 1:
        return float(value.reshape(-1)[0])
    return value

def load_dataset_cfg(path):
    with h5py.File(path, "r") as f:
        group = f["cfg"]
        return {k: _read_scalar(group[k]) for k in group.keys()
                if isinstance(group[k], h5py.Dataset)}

def get_train_global_scale(train_path, chunk_size=500):
    stats = {"dl": {"sum_sq": 0.0, "count": 0}, "ul": {"sum_sq": 0.0, "count": 0}}
    with h5py.File(train_path, "r") as f:
        for key, sn in [("csi_dl", "dl"), ("csi_ul", "ul")]:
            ds = f[key]; N = ds.shape[3]
            for start in range(0, N, chunk_size):
                chunk = ds[:, :, :, start:min(start+chunk_size, N)]
                r = chunk["real"].astype(np.float32)
                im = chunk["imag"].astype(np.float32)
                stats[sn]["sum_sq"] += float(np.sum(r**2) + np.sum(im**2))
                stats[sn]["count"]  += r.size + im.size
    out = {}
    for k in ("dl", "ul"):
        var = stats[k]["sum_sq"] / max(stats[k]["count"], 1)
        out[k] = {"std": float(np.sqrt(var + 1e-12))}
    print("Scale:", out)
    return out

class CSIDatasetManager:
    def __init__(self, train_path, val_path, test_path, stats):
        self.stats = stats
        self.paths = {"train": train_path, "val": val_path, "test": test_path}
        self.files = {}; self.datasets = {}; self.lengths = {}
        for split, path in self.paths.items():
            h = h5py.File(path, "r")
            self.files[split] = h
            self.datasets[split] = {"dl": h["csi_dl"], "ul": h["csi_ul"]}
            self.lengths[split]  = int(h["csi_dl"].shape[3])
            print(f"{split}: {self.lengths[split]} samples")

    def close(self):
        for h in self.files.values(): h.close()

    def _normalize(self, arr, k): return arr / (self.stats[k]["std"] + 1e-8)
    def denormalize(self, t,    k): return t   * (self.stats[k]["std"] + 1e-8)

    def _process(self, arr, k, normalize=True):
        r = arr["real"].astype(np.float32)
        im = arr["imag"].astype(np.float32)
        if normalize:
            r = self._normalize(r, k); im = self._normalize(im, k)
        return np.transpose(np.squeeze(np.stack([r, im], axis=2), axis=3), (3, 2, 0, 1))

    def get_batch(self, split, indices, snr_values=None):
        indices = np.sort(np.asarray(indices, dtype=np.int64))
        dl = torch.from_numpy(self._process(self.datasets[split]["dl"][:,:,:,indices], "dl")).float()
        ul = torch.from_numpy(self._process(self.datasets[split]["ul"][:,:,:,indices], "ul")).float()
        if snr_values is None:
            snr_values = np.random.uniform(cfg.snr_low, cfg.snr_high,
                                           size=(len(indices), 1)).astype(np.float32)
        else:
            snr_values = np.asarray(snr_values, dtype=np.float32).reshape(len(indices), 1)
        return dl, ul, torch.from_numpy(snr_values).float()

    def iterate_split(self, split, batch_size, shuffle=False, generator=None, fixed_snr=None):
        total = self.lengths[split]
        order = np.arange(total, dtype=np.int64)
        if shuffle:
            rng = generator if generator is not None else np.random.default_rng()
            rng.shuffle(order)
        for start in range(0, total, batch_size):
            bi = order[start:start+batch_size]
            sv = None if fixed_snr is None else np.full((len(bi),1), fixed_snr, dtype=np.float32)
            yield self.get_batch(split, bi, snr_values=sv)

In [ ]:
# Set these paths to the QuaDRiGa CSI .mat files on your machine.
train_file = "train_data.mat"
val_file   = "val_data.mat"
test_file  = "test_data.mat"


In [ ]:
stats = get_train_global_scale(train_file)
dataset = CSIDatasetManager(train_file, val_file, test_file, stats)


## AF Module

Channel-wise SNR-aware feature recalibration: GAP over (H,W), concat with SNR (dB), 2-layer MLP → sigmoid → per-channel scale.

In [ ]:
class AFModule(nn.Module):
    def __init__(self, channels, reduction_ratio=2):
        super().__init__()
        h = max(channels // reduction_ratio, 1)
        self.fc1 = nn.Linear(channels + 1, h)
        self.fc2 = nn.Linear(h, channels)

    def forward(self, x, snr):
        p = F.adaptive_avg_pool2d(x, 1).flatten(1)
        s = torch.sigmoid(self.fc2(F.relu(self.fc1(torch.cat([p, snr], dim=1)))))
        return x * s.view(x.size(0), x.size(1), 1, 1)

## ATN — Analysis Transform Network

Three-layer (or wider, in deeper variants) conv stack with asymmetric strides that compresses the 32×32 angular-delay map to the truncated representation used by the SC-CSI encoder.

In [ ]:
class ATN(nn.Module):
    """
    4-layer analysis transform network.

    Spatial compression schedule (H dimension of CSI, W stays fixed):
      layer 1: stride (2,1)  → H/2
      layer 2: stride (1,1)  → H/2   [NEW: refinement without compression]
      layer 3: stride (2,1)  → H/4
      layer 4: stride (2,1)  → H/8   → final T size matches truncated TAD domain
    """
    def __init__(self, ch=None):
        super().__init__()
        ch = ch or cfg.atn_channels   # 32

        self.conv1  = nn.Conv2d(2, ch, 3, stride=(2,1), padding=1)
        self.bn1    = nn.BatchNorm2d(ch); self.prelu1 = nn.PReLU(); self.af1 = AFModule(ch)

        # NEW refinement layer — same resolution, gathers wider context
        self.conv2  = nn.Conv2d(ch, ch, 3, stride=(1,1), padding=1)
        self.bn2    = nn.BatchNorm2d(ch); self.prelu2 = nn.PReLU(); self.af2 = AFModule(ch)

        self.conv3  = nn.Conv2d(ch, ch, 3, stride=(2,1), padding=1)
        self.bn3    = nn.BatchNorm2d(ch); self.prelu3 = nn.PReLU(); self.af3 = AFModule(ch)

        self.conv4  = nn.Conv2d(ch, 2, 3, stride=(2,1), padding=1)
        self.bn4    = nn.BatchNorm2d(2)

    def forward(self, x, snr):
        x = self.af1(self.prelu1(self.bn1(self.conv1(x))), snr)
        x = self.af2(self.prelu2(self.bn2(self.conv2(x))), snr)  # refinement
        x = self.af3(self.prelu3(self.bn3(self.conv3(x))), snr)
        x = self.bn4(self.conv4(x))
        return x

## Encoder — CSINet+ encoder with AF modules

Two 7×7 conv blocks with AF modules, then a fully-connected layer projects the flattened map to the M-dimensional real-valued codeword.

In [ ]:
class CsiNetPlusEncoderWithAF(nn.Module):
    def __init__(self, compression_ratio):
        super().__init__()
        self.total_elements = 2*32*32; self.M = self.total_elements // compression_ratio
        self.conv1=nn.Conv2d(2,2,7,padding=3); self.bn1=nn.BatchNorm2d(2); self.af1=AFModule(2)
        self.conv2=nn.Conv2d(2,2,7,padding=3); self.bn2=nn.BatchNorm2d(2); self.af2=AFModule(2)
        self.fc=nn.Linear(self.total_elements, self.M)
    def forward(self,x,snr):
        x=self.af1(F.leaky_relu(self.bn1(self.conv1(x)),0.3),snr)
        x=self.af2(F.leaky_relu(self.bn2(self.conv2(x)),0.3),snr)
        return self.fc(x.flatten(1))

## Real → complex symbols + power normalisation

Splits the M real outputs into an M/2-length complex vector and rescales it to unit average power per symbol.

In [ ]:
def enc_to_complex_and_normalize(e):
    k=e.shape[1]//2; s=torch.complex(e[:,:k],e[:,k:])
    return s/torch.sqrt(torch.mean(s.abs().square(),dim=1,keepdim=True)+1e-8)

## Wireless channel

Differentiable OFDM AWGN channel: picks `k` uplink subcarriers, transmits the complex symbols, adds Gaussian noise scaled to the requested SNR, and applies maximum-ratio combining at the BS.

In [ ]:
class WirelessChannelSimulator(nn.Module):
    def __init__(self, num_bs_antennas=32, trs=True):
        super().__init__(); self.Nt=num_bs_antennas; self.trs=trs
    def _sel(self,ns,k,dev):
        if self.training and self.trs: return torch.randperm(ns,device=dev)[:k]
        return torch.linspace(0,ns-1,k,device=dev).round().long()
    def forward(self,s,snr_db,h_ul):
        bs,k=s.shape; dev=s.device; idx=self._sel(h_ul.shape[2],k,dev)
        h_sliced=h_ul[:,:,idx,:]; h_u=torch.complex(h_sliced[:,0],h_sliced[:,1])
        nstd=torch.sqrt(1.0/torch.pow(10.,snr_db/10.)/2.).unsqueeze(-1)
        z=torch.complex(torch.randn(bs,k,self.Nt,device=dev)*nstd,
                        torch.randn(bs,k,self.Nt,device=dev)*nstd)
        y=h_u*s.unsqueeze(-1)+z; w=h_u/(torch.norm(h_u,dim=2,keepdim=True)+1e-8)
        return torch.sum(torch.conj(w)*y,dim=2)

## C2R — Complex → real for the decoder

In [ ]:
class ComplexToReal(nn.Module):
    def forward(self,s): return torch.cat([s.real,s.imag],dim=1)

## Decoder — CSINet+ RefineNet stack

FC → 32×32 feature map, an initial conv block, then a chain of RefineNet residual blocks (each conv block is followed by an AF module).

In [ ]:
class ModifiedRefineNetBlock(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.c1=nn.Conv2d(ch,8,7,padding=3);  self.b1=nn.BatchNorm2d(8);  self.a1=AFModule(8)
        self.c2=nn.Conv2d(8,16,5,padding=2);  self.b2=nn.BatchNorm2d(16); self.a2=AFModule(16)
        self.c3=nn.Conv2d(16,ch,3,padding=1); self.b3=nn.BatchNorm2d(ch); self.a3=AFModule(ch)
    def forward(self,x,snr):
        r=x; x=self.a1(F.leaky_relu(self.b1(self.c1(x)),0.3),snr)
        x=self.a2(F.leaky_relu(self.b2(self.c2(x)),0.3),snr); x=self.a3(self.b3(self.c3(x)),snr)
        return r+x

In [ ]:
class CsiNetPlusDecoder(nn.Module):
    def __init__(self,input_dim,height=32,width=32,channels=2,num_blocks=5):
        super().__init__()
        self.h=height; self.w=width; self.ch=channels; self.flat=height*width*channels
        self.fc=nn.Linear(input_dim,self.flat)
        self.ic=nn.Conv2d(channels,channels,7,padding=3); self.ib=nn.BatchNorm2d(channels); self.ia=AFModule(channels)
        self.chain=nn.ModuleList([ModifiedRefineNetBlock(channels) for _ in range(num_blocks)])
    def forward(self,x,snr):
        x=self.fc(x).view(-1,self.ch,self.h,self.w)
        x=self.ia(F.leaky_relu(self.ib(self.ic(x)),0.3),snr)
        for b in self.chain: x=b(x,snr)
        return x

## STN — Synthesis Transform Network

Mirror image of the ATN: transposed-conv stack that expands the latent back to the 32×32 angular-delay map.

In [ ]:
class STN(nn.Module):
    """
    4-layer synthesis transform network (mirrors ATN exactly in reverse).

    Upsampling schedule:
      layer 1: stride (2,1)  → x2 in H
      layer 2: stride (2,1)  → x4 in H
      layer 3: stride (1,1)  → same (refinement, mirrors ATN layer 2)  [NEW]
      layer 4: stride (2,1)  → x8 in H → original CSI size
    """
    def __init__(self, ch=None):
        super().__init__()
        ch = ch or cfg.atn_channels   # 32

        self.tc1    = nn.ConvTranspose2d(2, ch, 3, stride=(2,1), padding=1, output_padding=(1,0))
        self.bn1    = nn.BatchNorm2d(ch); self.p1 = nn.PReLU(); self.af1 = AFModule(ch)

        self.tc2    = nn.ConvTranspose2d(ch, ch, 3, stride=(2,1), padding=1, output_padding=(1,0))
        self.bn2    = nn.BatchNorm2d(ch); self.p2 = nn.PReLU(); self.af2 = AFModule(ch)

        # NEW refinement layer
        self.tc3    = nn.ConvTranspose2d(ch, ch, 3, stride=(1,1), padding=1)
        self.bn3    = nn.BatchNorm2d(ch); self.p3 = nn.PReLU(); self.af3 = AFModule(ch)

        self.tc4    = nn.ConvTranspose2d(ch, 2, 3, stride=(2,1), padding=1, output_padding=(1,0))
        self.bn4    = nn.BatchNorm2d(2)

    def forward(self, x, snr):
        x = self.af1(self.p1(self.bn1(self.tc1(x))), snr)
        x = self.af2(self.p2(self.bn2(self.tc2(x))), snr)
        x = self.af3(self.p3(self.bn3(self.tc3(x))), snr)  # refinement
        x = self.bn4(self.tc4(x))
        return x

## Build the modules

In [ ]:
dataset_cfg = load_dataset_cfg(train_file)
atn = ATN().to(device)
encoder = CsiNetPlusEncoderWithAF(compression_ratio=cfg.compression_ratio).to(device)
channel_sim = WirelessChannelSimulator(num_bs_antennas=int(dataset_cfg['num_bs_antennas'])).to(device)
c2r = ComplexToReal().to(device)
decoder = CsiNetPlusDecoder(input_dim=encoder.M).to(device)
stn = STN().to(device)

all_params = (list(atn.parameters()) + list(encoder.parameters())
              + list(decoder.parameters()) + list(stn.parameters()))
print('Total trainable parameters:', sum(p.numel() for p in all_params if p.requires_grad))


## Training loop

In [ ]:
def samplewise_linear_nmse(ht,hp):
    return torch.sum((ht-hp)**2,dim=(1,2,3))/(torch.sum(ht**2,dim=(1,2,3))+1e-8)

def nmse_db_from_sums(e,p): return 10.*math.log10((e/max(p,1e-12))+1e-12)

def forward_pass(H_d,H_u,snr):
    T=atn(H_d,snr); c=encoder(T,snr); s=enc_to_complex_and_normalize(c)
    s_hat=channel_sim(s,snr,H_u); c_hat=c2r(s_hat); T_hat=decoder(c_hat,snr)
    return stn(T_hat,snr)

def compute_loss(H_true,H_pred,epoch_index):
    mse=mse_criterion(H_pred,H_true); nmse=samplewise_linear_nmse(H_true,H_pred).mean()
    total=mse if epoch_index<cfg.warmup_epochs else cfg.mse_weight*mse+cfg.nmse_weight*nmse
    return total,mse.detach(),nmse.detach()

def run_epoch(split,epoch_index=0,fixed_snr=None):
    is_train=split=="train"
    for m in [atn,encoder,decoder,stn,channel_sim]: m.train(is_train)
    rng=np.random.default_rng(SEED+epoch_index)
    total_loss=total_mse=total_samples=0.; error_sum=power_sum=0.
    for bi,(H_d,H_u,snr) in enumerate(
        dataset.iterate_split(split,cfg.batch_size,shuffle=is_train,
                              generator=rng if is_train else None,fixed_snr=fixed_snr)):
        if is_train and debug_train_batches and bi>=debug_train_batches: break
        if not is_train and debug_eval_batches and bi>=debug_eval_batches: break
        H_d=H_d.to(device,non_blocking=True); H_u=H_u.to(device,non_blocking=True); snr=snr.to(device,non_blocking=True)
        if is_train: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_train):
            H_hat=forward_pass(H_d,H_u,snr); loss,ml,nl=compute_loss(H_d,H_hat,epoch_index)
            if is_train:
                loss.backward()
                if cfg.grad_clip>0: torch.nn.utils.clip_grad_norm_(all_params,cfg.grad_clip)
                optimizer.step()
        bs=H_d.size(0); total_loss+=float(loss.detach())*bs; total_mse+=float(ml)*bs; total_samples+=bs
        Hd_d=dataset.denormalize(H_d.detach(),"dl"); Hh_d=dataset.denormalize(H_hat.detach(),"dl")
        error_sum+=float(torch.sum((Hd_d-Hh_d)**2)); power_sum+=float(torch.sum(Hd_d**2))
    return {"loss":total_loss/max(total_samples,1),"mse":total_mse/max(total_samples,1),
            "nmse_db":nmse_db_from_sums(error_sum,power_sum),
            "linear_nmse":error_sum/max(power_sum,1e-12),"samples":int(total_samples)}

def save_checkpoint(epoch,best,tag):
    torch.save({"epoch":epoch,"cfg":cfg.__dict__,"stats":stats,"best_linear_nmse":best,
                "atn_state_dict":atn.state_dict(),"encoder_state_dict":encoder.state_dict(),
                "decoder_state_dict":decoder.state_dict(),"stn_state_dict":stn.state_dict(),
                "optimizer_state_dict":optimizer.state_dict(),"scheduler_state_dict":scheduler.state_dict()},
               Path(cfg.checkpoint_dir)/tag); print(f"Saved: {Path(cfg.checkpoint_dir)/tag}")

def find_latest_checkpoint():
    d=Path(cfg.checkpoint_dir); cands=[*d.glob("epoch_*.pth"),d/"final_model.pth",d/"best_model.pth"]
    cands=[p for p in cands if p.exists()]
    if not cands: raise FileNotFoundError(f"No ckpts in {d}")
    return max(cands,key=lambda p:p.stat().st_mtime)

def load_checkpoint(path):
    ck=torch.load(path,map_location=device)
    atn.load_state_dict(ck["atn_state_dict"]); encoder.load_state_dict(ck["encoder_state_dict"])
    decoder.load_state_dict(ck["decoder_state_dict"]); stn.load_state_dict(ck["stn_state_dict"])
    if "optimizer_state_dict" in ck: optimizer.load_state_dict(ck["optimizer_state_dict"])
    if "scheduler_state_dict" in ck: scheduler.load_state_dict(ck["scheduler_state_dict"])
    print(f"Loaded: {path}"); return ck

def evaluate_snr_sweep(snr_points,split="test"):
    results=[]
    for snr_db in snr_points:
        m=run_epoch(split,fixed_snr=snr_db); results.append(m["nmse_db"])
        print(f"SNR {snr_db:>4} dB -> NMSE {m['nmse_db']:.3f} dB")
    return results

### Optimiser, scheduler and loss weights

These cells reproduce the variant's exact training recipe — open the source `.py` for the line-by-line argparse / CLI logic.

In [ ]:
# optimizer = optim.Adam(all_params, lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
# scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min',
#                                                   factor=0.5, patience=cfg.patience,
#                                                   min_lr=cfg.min_lr)
# mse_criterion = nn.MSELoss()
# (See the .py for any variant-specific overrides — e.g. cosine LR
#  schedules, AdamW, or per-parameter-group weight decay.)


### Run training

```python
for epoch in range(cfg.epochs):
    train_metrics = run_epoch('train', epoch_index=epoch)
    val_metrics   = run_epoch('val',   epoch_index=epoch)
    # scheduler.step(val_metrics['linear_nmse'])
```

After training, sweep test NMSE over a fixed SNR grid:

```python
snr_points = [-10, -5, 0, 5, 10]
nmse_db = evaluate_snr_sweep(snr_points, split='test')
```